In [1]:
%pip install keybert transformers sentence_transformers nltk pandas model2vec model2vec[distill]

Note: you may need to restart the kernel to use updated packages.


In [10]:
# https://www.imt-journal.ru/archive/public/article?id=381
test_str = "В работе исследуется тематическое многообразие междисциплинарного журнала. " \
    "Цель исследований составляет построение графа знаний журнала для тематического представления и систематизации электронного архива и новых публикаций журнала. " \
    "Исходные данные представляют собой статьи журнала, посвященные различным информационным и математическим технологиям в науке и управлении, то есть междисциплинарным исследованиям. " \
    "Предлагается систематизация текстов с помощью методов векторного анализа. В процессе тематического анализа контента журнала предлагается разбиение на рубрики, устанавливаются связи рубрик и статей с соответствующими описаниями специальностей ВАК. " \
    "Для анализа тематики используется разведочный анализ исходных текстов, далее применяются методы интеллектуального анализа данных. Результаты разбиения предоставляются экспертам журнала, после чего вырабатывается решение о формировании тематической рубрики и включении в нее специальностей ВАК. " \
    "Статьи журнала интегрируются в семантическую библиотеку LibMeta, в силу чего онтология библиотеки достраивается и формируется онтология журнала, и на этой основе строится граф знаний журнала. " \
    "Предлагается процедура навигации по контенту журнала с помощью графа знаний в семантической библиотеке LibMeta, которая может стать основой для информационного сопровождения научных исследований и создания цифрового ассистента в междисциплинарной предметной области. " \
    "Примеры приведены для конкретного контента журнала, но предложенная технология может быть распространена на другие журналы, так как большинство журналов, относящихся к нескольким специальностям ВАК, естественным образом захватывают несколько дисциплин."

In [9]:
# https://github.com/MaartenGr/BERTopic/issues/897#issuecomment-1374387291
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
nltk.download('stopwords')
my_stopwords = CountVectorizer(stop_words=stopwords.words("russian"))

[nltk_data] Downloading package stopwords to /home/alexey/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
from keybert import KeyBERT


sci_rus_small=KeyBERT('mlsa-iai-msu-lab/sci-rus-small') # larger then tiny
frida_model=KeyBERT('ai-forever/FRIDA')
embeddinggemma_model=KeyBERT('unsloth/embeddinggemma-300m') # same as google/embeddinggemma-300m
bge_m3_model=KeyBERT('BAAI/bge-m3')
qwen3embedding_model=KeyBERT('Qwen/Qwen3-Embedding-0.6B')

In [ ]:
from model2vec.distill import distill

# Distill a Sentence Transformer model, in this case the BAAI/bge-base-en-v1.5 model
m2v_model = distill(model_name="unsloth/embeddinggemma-300m")

# Save the model
m2v_model.save_pretrained("m2v_model")

Encoding tokens: 100%|██████████| 255732/255732 [01:12<00:00, 3504.51 tokens/s]


In [14]:
from keybert.backend import BaseEmbedder
from model2vec import StaticModel
from keybert import KeyBERT

class Model2Vec_gemma(BaseEmbedder):
    def __init__(self, embedding_model="m2v_model"):
        super().__init__()
        self.embedding_model = StaticModel.from_pretrained(embedding_model)

    def embed(self, documents, verbose=False):
        embeddings = self.embedding_model.encode(documents, show_progress_bar=verbose)
        return embeddings

# Pass custom backend to keybert
gemma_m2v_model = KeyBERT(model=Model2Vec_gemma())

In [15]:
gemma_m2v_model.extract_keywords("Hello world this is a test sentence", top_n=5)

[('sentence', 0.9137), ('hello', 0.9068), ('test', 0.8924), ('world', 0.8823)]

In [17]:
potion_128m = KeyBERT(model=Model2Vec_gemma("minishlab/potion-multilingual-128M"))

model.safetensors:   0%|          | 0.00/512M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

In [18]:
import pandas as pd
from IPython.display import HTML, display

def extract_keywords(text, kw_model, num_keywords=20):
    return kw_model.extract_keywords(f"{text}", keyphrase_ngram_range=(2, 2), top_n=num_keywords, stop_words=my_stopwords.stop_words)

pd_res = []
for model_name, model in [
    ("sci_rus_small", sci_rus_small),
    ("frida_model", frida_model),
    ("embeddinggemma_model", embeddinggemma_model),
    ("bge_m3_model", bge_m3_model),
    ("qwen3embedding_model", qwen3embedding_model),
    ("gemma_m2v_model", gemma_m2v_model),
    ("potion_128m", potion_128m),
]:
    keywords = extract_keywords(test_str, model, num_keywords=10)
    pd_res.append({"model_name": model_name, "keywords": ", ".join([kw for kw, score in keywords])})
display(HTML(pd.DataFrame(pd_res).to_html()))

,model_name,keywords
0,sci_rus_small,"междисциплинарного журнала, другие журналы, исследуется тематическое, журнала предлагается, журнала тематического, междисциплинарным исследованиям, статьи журнала, онтология журнала, экспертам журнала, интегрируются семантическую"
1,frida_model,"междисциплинарного журнала, журнала тематического, семантическую библиотеку, семантической библиотеке, онтология журнала, предлагается систематизация, междисциплинарной предметной, статьи журнала, тематической рубрики, анализа тематики"
2,embeddinggemma_model,"междисциплинарного журнала, журнала тематического, междисциплинарным исследованиям, анализа тематики, междисциплинарной предметной, журналов относящихся, анализа контента, контенту журнала, библиотеке libmeta, многообразие междисциплинарного"
3,bge_m3_model,"междисциплинарного журнала, онтология журнала, междисциплинарным исследованиям, анализа тематики, исследуется тематическое, журнала тематического, тематического анализа, журнала интегрируются, журнала вырабатывается, междисциплинарной предметной"
4,qwen3embedding_model,"междисциплинарного журнала, знаний журнала, журнала тематического, тематического анализа, междисциплинарным исследованиям, анализа тематики, статьи журнала, тематической рубрики, онтология журнала, исследуется тематическое"
5,gemma_m2v_model,"междисциплинарным исследованиям, междисциплинарной предметной, междисциплинарного журнала, управлении междисциплинарным, интегрируются семантическую, ассистента междисциплинарной, семантическую библиотеку, навигации контенту, многообразие междисциплинарного, приведены конкретного"
6,potion_128m,"журнала тематического, журнала интегрируются, журнала вырабатывается, журнала основе, анализа тематики, журнала предложенная, контента журнала, знаний журнала, онтология журнала, публикаций журнала"
